# Progetto Smart Vehicular Systems - Multi-modal black box recorder
  - Gabriele Arcese
  - Diego Colì

## Setup

In [ ]:
%pip install paho-mqtt
import carla
import time
import random
import socket
import threading
import queue
from dataclasses import dataclass, field, asdict
from typing import Optional
import numpy as np
import json
import paho.mqtt.client as mqtt

try:
    import pygame
except ImportError:
    pygame = None
    
try:
    import cv2
    import numpy as np
except ImportError:
    cv2 = None
    np = None

pygame 2.6.1 (SDL 2.28.4, Python 3.7.12)
Hello from the pygame community. https://www.pygame.org/contribute.html


## Connessione al client

In [2]:
client = carla.Client("localhost", 2000)
client.set_timeout(15.0)
world = client.get_world()
spectator = world.get_spectator()

print(f"Client version: {client.get_client_version()}")
print(f"Server version: {client.get_server_version()}")

Client version: 0.9.15
Server version: 0.9.15


## MQTT client

In [ ]:
# queue thread-safe per accumulare messaggi ricevuti
mqtt_msgs_q = queue.Queue()

def _on_connect(client, userdata, flags, rc):
    if rc == 0:
        client.connected_flag = True
        # sottoscrivi i topic desiderati
        client.subscribe([("recorder/control", 0), ("vehicle/+/status", 0)])
    else:
        client.connected_flag = False

def _on_message(client, userdata, msg):
    ts = time.time()
    payload = msg.payload
    # prova a decodificare json, fallback a stringa/bytes
    parsed = None
    try:
        parsed = json.loads(payload.decode("utf-8"))
    except Exception:
        try:
            parsed = payload.decode("utf-8")
        except Exception:
            parsed = payload  # raw bytes
    record = {"topic": msg.topic, "payload": parsed, "qos": msg.qos, "ts": ts}
    try:
        mqtt_msgs_q.put_nowait(record)
    except queue.Full:
        pass

def start_mqtt(broker="localhost", port=1883, keepalive=60):
    client = mqtt.Client()
    client.on_connect = _on_connect
    client.on_message = _on_message
    client.connected_flag = False
    client.connect(broker, port, keepalive=keepalive)
    client.loop_start()   # avvia loop in background thread
    return client

def stop_mqtt(client):
    if client is None: return
    try:
        client.loop_stop()
        client.disconnect()
    except Exception:
        pass

# Helper: svuota la queue e ritorna lista (da chiamare nel tick)
def drain_mqtt_queue():
    items = []
    while True:
        try:
            items.append(mqtt_msgs_q.get_nowait())
        except queue.Empty:
            break
    return items

## Dataclass EventRecord
Prodotto ad ogni tick della simulazione: è l'unità atomica del ring buffer.

| Campo | Sorgente CARLA | Tipo |
|---|---|---|
| `timestamp_sim` | `world.get_snapshot().timestamp.elapsed_seconds` | `float` |
| `timestamp_wall` | `time.time()` | `float` |
| `frame_id` | contatore incrementale | `int` |
| `camera_frames` | callback `sensor.camera.rgb` (una per camera) | `dict[str, np.ndarray]` |
| `location` | `vehicle.get_location()` | `carla.Location` |
| `velocity` | `vehicle.get_velocity()` | `carla.Vector3D` |
| `acceleration` | `vehicle.get_acceleration()` | `carla.Vector3D` |
| `control` | `vehicle.get_control()` | `carla.VehicleControl` |
| `warnings` | logica interna (LiDAR, collisione) | `list[str]` |
| `mqtt_messages` | thread MQTT | `list[str]` |
| `reasoning_text` | explanation agent | `str \| None` |
| `reasoning_ts` | wall-clock emissione agent | `float \| None` |

In [3]:
@dataclass
class VehicleState:
    """Stato cinematico e di controllo del veicolo in un istante."""
    x: float
    y: float
    z: float
    vx: float
    vy: float
    vz: float
    ax: float
    ay: float
    az: float
    throttle: float
    brake: float
    steer: float
    hand_brake: bool
    reverse: bool

@property
def speed_ms(self) -> float:
    """Velocità scalare in m/s."""
    return float(np.sqrt(self.vx**2 + self.vy**2 + self.vz**2))

@property
def speed_kmh(self) -> float:
    return self.speed_ms * 3.6

@dataclass
class EventRecord:
    """Unità atomica del ring buffer — un record per tick di simulazione."""
    # --- Temporali ---
    frame_id: int            # contatore tick, usato come chiave nel viewer
    timestamp_sim: float     # secondi simulazione (univoco per tick in sync mode)
    timestamp_wall: float    # wall-clock al momento della costruzione

    # --- Percezione ---
    vehicle_state: VehicleState

    # --- Camera (dict role → frame BGR; vuoto se nessun frame è ancora arrivato) ---
    # Esempio: {"front": np.ndarray, "rear": np.ndarray}
    camera_frames: dict = field(default_factory=dict, repr=False)

    # --- Logica e messaggistica ---
    warnings: list = field(default_factory=list)       # es. ["COLLISION_IMMINENT"]
    mqtt_messages: list = field(default_factory=list)  # payload raw ricevuti nel tick

    # --- Explanation agent ---
    reasoning_text: Optional[str] = None
    reasoning_ts: Optional[float] = None  # wall-clock di emissione dell'agente

def to_dict(self) -> dict:
    """Serializza tutto tranne camera_frames (salvati separatamente come JPEG)."""
    d = asdict(self)
    d.pop("camera_frames")  # i frame vanno in frames/<frame_id:06d>_<role>.jpg
    return d

def has_camera(self, role: str = None) -> bool:
    """True se almeno una camera (o la camera `role`) ha un frame nel tick."""
    if role:
        return role in self.camera_frames
    return len(self.camera_frames) > 0

def camera_roles(self) -> list:
    return list(self.camera_frames.keys())

def has_reasoning(self) -> bool:
    return self.reasoning_text is not None

Il **timestamp** di simulazione è la chiave di allineamento: tutto viene sincronizzato su di esso perché in modalità sincrona.

## VehicleState

In [4]:
# VehicleState partendo da un attore
def build_vehicle_state(vehicle) -> VehicleState:
    loc = vehicle.get_location()
    vel = vehicle.get_velocity()
    acc = vehicle.get_acceleration()
    ctrl = vehicle.get_control()
    return VehicleState(
    x=loc.x, y=loc.y, z=loc.z,
    vx=vel.x, vy=vel.y, vz=vel.z,
    ax=acc.x, ay=acc.y, az=acc.z,
    throttle=ctrl.throttle,
    brake=ctrl.brake,
    steer=ctrl.steer,
    hand_brake=ctrl.hand_brake,
    reverse=ctrl.reverse,
)

## Attivazione modalità sync
In modalità **sincrona**, il server avanza solo quando il client chiama `world.tick()`. Ciò è utile per esperimenti controllati e risultati riproducibili.


In [29]:
settings = world.get_settings()
settings.synchronous_mode = True
world.apply_settings(settings)
print("Modalità sincrona attivata")

Modalità sincrona attivata


## Spawn veicolo

In [ ]:
def move_spectator_to(transform, spectator, distance=7.0, x=0, y=0, z=3.0, yaw=0, pitch=-15, roll=0):
    back_location = transform.location - transform.get_forward_vector() * distance
    
    back_location.x += x
    back_location.y += y
    back_location.z += z
    transform.rotation.yaw += yaw
    transform.rotation.pitch = pitch
    transform.rotation.roll = roll
    
    spectator_transform = carla.Transform(back_location, transform.rotation)
    
    spectator.set_transform(spectator_transform)

def spawn_vehicle(world, vehicle_index=16, spawn_index=10):
    blueprint_library = world.get_blueprint_library()
    vehicle_bp = blueprint_library.filter('vehicle.*')[vehicle_index]
    spawn_point = world.get_map().get_spawn_points()[spawn_index]
    vehicle = world.spawn_actor(vehicle_bp, spawn_point)
    return vehicle

def draw_on_screen(world, transform, content="O", color=carla.Color(0, 255, 0), life_time=20):
    world.debug.draw_string(transform.location, content, color=color, life_time=life_time)


In [ ]:
vehicle = spawn_vehicle(world)
vehicle.set_autopilot(True)

## Ciclo di simulazione

In [ ]:
def tcp_check(host, port, timeout=2.0):
    s = socket.socket()
    s.settimeout(timeout)
    try:
        s.connect((host, port))
        s.close()
        return True, None
    except Exception as e:
        return False, str(e)

targets = [("localhost", 1883), ("test.mosquitto.org", 1883)]
for host, port in targets:
    ok, err = tcp_check(host, port)
    print(f"{host}:{port} -> {'OK' if ok else 'FAIL'}{'' if ok else ': '+err}")

# Try connecting with paho to localhost (shows MQTT error if refused)
client = mqtt.Client()
try:
    client.connect("localhost", 1883, keepalive=5)
    client.loop_start()
    time.sleep(0.3)
    client.loop_stop()
    print("paho-mqtt: connection to localhost:1883 succeeded")
except Exception as e:
    print("paho-mqtt: connection to localhost:1883 failed:", e)
    print("\nHints:")
    print("- If you want a local broker, start Mosquitto (outside notebook) or run Docker:")
    print("  docker run -d --name mosquitto -p 1883:1883 -p 9001:9001 eclipse-mosquitto")
    print("- Or switch to the public test broker: test.mosquitto.org:1883")

In [ ]:
try:
    mqtt_client = start_mqtt(broker="localhost", port=1883)
    while True:
        world.tick()
        msgs = drain_mqtt_queue()    # lista di dict (topic,payload,ts)
        record = EventRecord(
            frame_id=frame_id,
            timestamp_sim=world.get_snapshot().timestamp.elapsed_seconds,
            timestamp_wall=time.time(),
            vehicle_state=build_vehicle_state(vehicle),
            camera_frames={},  # o frames raccolti
            warnings=[],
            mqtt_messages=msgs,
        )
        deque.append(record)
        move_spectator_to(vehicle.get_transform(), spectator, distance=8.0, z=3.0, pitch=-20)
        time.sleep(0.05)  
except KeyboardInterrupt:
    stop_mqtt(mqtt_client)
    pass

In [33]:
settings.synchronous_mode = False
world.apply_settings(settings)
print("Modalità sincrona disattivata")

Modalità sincrona disattivata


In [34]:
vehicle.destroy()
print("Veicolo distrutto")
vehicle = None

Veicolo distrutto
